# Notebook 01 - Model A: property-only predictors
**A Machine Learning-Based Approach to Residential Property Tax Assessment in a Data-Constrained Environment:
Dematagoda Ward, Colombo Municipal Council**

Model A uses only the property characteristics recorded by CMC (building age, condition grade, total floor
area, number of floors) and **no location information**. It sets the baseline performance (methodology 3.8).
Model B (notebook 02) adds the OSM/GIS distances; notebook 03 compares the two.

* Target: **CMC-recorded Annual Value (2025)**. It is not market value.
* Algorithms: Multiple Linear Regression (baseline), Random Forest, XGBoost.
* Validation: a fixed 80:20 split (seed 42, stratified by Annual Value). Tuning and 5-fold CV (repeated 10 times) use
  only the training set. The test set is used once, at the end.
* All models are fitted on **log(Annual Value)**; predictions are converted back to rupees, and every metric is in rupees.

This notebook also contains the **dataset description** (Chapter 4.2).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from scipy import stats

from src import config as C, data as D, modelling as M, runner as R, plots as P, interpret as I
P.setup_style()
pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", lambda v: f"{v:,.3f}")
T, F = C.TABLES_DIR, C.FIGURES_DIR
def show(fig, name):
    P.save(fig, name); plt.show()

## 1. Load and clean the data (methodology 3.6-3.7)

In [2]:
data = D.load_and_prepare()
lab, train, test = data["labelled"], data["train"], data["test"]
(T / "AB_cleaning_log.txt").write_text("\n".join(data["log"]) + "\n")
data["accounting"].to_csv(T / "AB_record_accounting.csv", index=False)
data["accounting"]

 - Raw file: 566 records x 22 columns.
 - 'Annual_Value': 3 literal 'NA' entries treated as missing.
 - Numeric coercion: 0 malformed numeric value(s) set to missing.
 - Condition: 'Fair +' -> 'Fair+'; 'Modern' and 'Good (Modern)' (n=4) merged into 'Good'; grade order Poor < Fair- < Fair < Fair+ < Good.
 - 'First_Floor_Area': 350 blank value(s) set to 0 (no such floor).
 - 'Second_Floor_Area': 461 blank value(s) set to 0 (no such floor).
 - 'Third_Floor_Area': 505 blank value(s) set to 0 (no such floor).
 - Invalid values (outside valid range): 0 set to missing.
 - Duplicates: 124 records share an assessment number with another record; 0 exact duplicate(s) (all attributes and distances identical) removed.
 - Records without Annual Value: 64 (kept separately as unlabelled).
 - Labelled records without floor area: 0 (excluded).
 - Modelling dataset: 502 labelled records.
 - Train / test: 401 / 101 (fixed split saved to split_ids.csv)


,Step,Records
0,Records in raw file,566
1,Exact duplicates removed,0
2,No Annual Value (unlabelled),64
3,Annual Value but no floor area,0
4,Records used for modelling,502


### Missing values in the raw file

In [3]:
miss = D.raw_missing_table(data["raw"])
miss = miss[miss["Variable"].isin(["Age", "Condition", "Ground_Floor_Area", "First_Floor_Area", "Second_Floor_Area",
                                   "Third_Floor_Area", "Annual_Value"] + C.GIS_FEATURES)]
# missingness among the labelled (modelling) records
lab_miss = lab[["Age", "Condition", "Total_Floor_Area"]].isna().sum().rename("Missing in modelling data")
miss.to_csv(T / "AB_missing_values.csv", index=False)
display(miss)
lab_miss.to_frame()

,Variable,Records,Missing,Missing (%),Literal 'NA' entries
3,Age,566,62,11.000,0
4,Condition,566,63,11.100,0
5,Ground_Floor_Area,566,56,9.900,0
6,First_Floor_Area,566,406,71.700,0
7,Second_Floor_Area,566,517,91.300,0
8,Third_Floor_Area,566,561,99.100,0
9,Annual_Value,566,64,11.300,3
11,Dist_AB_Road,566,0,0.000,0
12,Dist_Bank,566,0,0.000,0
13,Dist_Bus_Stop,566,0,0.000,0


,Missing in modelling data
Age,4
Condition,5
Total_Floor_Area,0


**Interpretation.** Upper-floor fields are blank when a building has no such floor, so a blank is read as 0
(the rule from the interim prototype, to be confirmed with CMC). No predictor in the modelling data is above the 30 %
exclusion threshold. The few missing ages and condition grades of labelled properties are imputed inside each model
pipeline (median / mode), using training data only.

### Condition grades and repeated assessment numbers

In [4]:
data["condition_table"].to_csv(T / "AB_condition_cleaning.csv", index=False)
display(data["condition_table"])
dr = data["duplicate_review"]
dr.to_csv(T / "AB_duplicate_assessment_numbers.csv", index=False)
dr["Classification"].value_counts().to_frame("Records")

,Raw value,Cleaned grade,Records
0,(blank),(missing),63
1,Fair,Fair,169
2,Fair +,Fair+,33
3,Fair+,Fair+,178
4,Fair-,Fair-,20
5,Good,Good,93
6,Good (Modern),Good,2
7,Modern,Good,2
8,Poor,Poor,6


,Records
Classification,
Same number at a different location (number re-used on another road),124


Assessment numbers repeat because the same house number exists on different roads. The records share a number
but have different locations, attributes and Unique Property IDs, so all are retained. There are no exact duplicate
records.

## 2. Dataset description (Chapter 4.2)

In [5]:
desc_vars = [C.TARGET, "Total_Floor_Area", "Ground_Floor_Area", "Num_Floors", "Age", "AV_per_sqft"]
rows = []
for v in desc_vars:
    s = lab[v].dropna()
    rows.append({"Variable": C.label(v), "n": len(s), "Mean": s.mean(), "Median": s.median(), "SD": s.std(),
                 "Min": s.min(), "Max": s.max(), "Skewness": stats.skew(s)})
desc = pd.DataFrame(rows)
desc.to_csv(T / "A_descriptive_statistics.csv", index=False)
desc

,Variable,n,Mean,Median,SD,Min,Max,Skewness
0,CMC Annual Value (Rs.),502,"25,971.697","16,800.000","28,409.869","1,152.000","307,200.000",4.042
1,Total floor area (sq ft),502,993.171,657.000,"1,120.431",143.000,"12,238.000",4.744
2,Ground floor area (sq ft),502,693.163,550.000,533.342,143.000,"3,704.000",1.867
3,Number of floors,502,1.422,1.000,0.693,1.000,4.000,1.523
4,Building age (years),498,25.661,25.000,11.058,3.000,80.000,1.258
5,Annual Value per sq ft (Rs.),502,26.302,26.067,8.350,2.224,114.000,5.823


In [6]:
cond = (lab.groupby("Condition")
          .agg(n=(C.TARGET, "size"), Median_AV=(C.TARGET, "median"), Median_floor_area=("Total_Floor_Area", "median"),
               Median_AV_per_sqft=("AV_per_sqft", "median"))
          .reindex([c for c in C.CONDITION_ORDER if c in set(lab["Condition"])]).reset_index())
cond.to_csv(T / "A_condition_summary.csv", index=False)
floors = lab.groupby("Num_Floors").agg(n=(C.TARGET, "size"), Median_AV=(C.TARGET, "median")).reset_index()
floors.to_csv(T / "A_floors_summary.csv", index=False)
display(cond); floors

,Condition,n,Median_AV,Median_floor_area,Median_AV_per_sqft
0,Poor,4,"4,560.000",256.000,17.669
1,Fair-,20,"5,760.000",301.500,19.200
2,Fair,168,"7,440.000",360.000,25.756
3,Fair+,210,"19,200.000",792.000,25.309
4,Good,95,"42,240.000","1,520.000",28.087


,Num_Floors,n,Median_AV
0,1.000,344,"13,440.000"
1,2.000,109,"28,800.000"
2,3.000,44,"30,720.000"
3,4.000,5,"28,800.000"


In [7]:
rows = []
for v in ["Total_Floor_Area", "Num_Floors", "Age"]:
    s = lab[[v, C.TARGET]].dropna()
    rho, p = stats.spearmanr(s[v], s[C.TARGET])
    r, pr = stats.pearsonr(np.log(s[v]) if v == "Total_Floor_Area" else s[v], np.log(s[C.TARGET]))
    rows.append({"Variable": C.label(v), "Spearman rho with AV": rho, "p-value": p,
                 "Pearson r (log AV)" + (" vs log area" if v == "Total_Floor_Area" else ""): r})
prop_corr = pd.DataFrame(rows)
prop_corr.to_csv(T / "A_property_correlations.csv", index=False)
prop_corr

,Variable,Spearman rho with AV,p-value,Pearson r (log AV) vs log area,Pearson r (log AV)
0,Total floor area (sq ft),0.957,0.000,0.953,NaN
1,Number of floors,0.428,0.000,NaN,0.417
2,Building age (years),-0.447,0.000,NaN,-0.319


In [8]:
show(P.annual_value_distribution(lab), "4_1_annual_value_distribution.png")
show(P.property_relationships(lab), "4_2_property_relationships.png")

**Interpretation.** Annual Value is strongly right-skewed: a few large multi-storey houses have values far
above the median. The log scale (panel b) is close to symmetric, which is why the models use log(Annual Value). Floor
area is by far the strongest single correlate. Annual Value per sq ft is quite stable, which is consistent with CMC
applying a rate per sq ft. Its level rises with the condition grade, so condition explains part of the variation in
rate.

In [9]:
flag = lab[lab["Flag_Extreme_AV_per_sqft"] == 1][["Property_ID", "Assessment_No", "Total_Floor_Area", C.TARGET,
                                               "AV_per_sqft", "Partition"]]
flag.to_csv(T / "AB_flagged_extreme_rate_records.csv", index=False)
flag

,Property_ID,Assessment_No,Total_Floor_Area,Annual_Value,AV_per_sqft,Partition
132,DEM_00133,117/1 3/1,160.000,"18,240.000",114.000,train
311,DEM_00312,118 1/1,"12,238.000","57,600.000",4.707,train
331,DEM_00332,146/20A,"1,150.000","65,280.000",56.765,test
377,DEM_00378,220,750.000,"82,720.000",110.293,train
382,DEM_00383,98/10,654.000,"6,240.000",9.541,train
389,DEM_00390,105/6,518.000,"1,152.000",2.224,test
523,DEM_00524,16,900.000,"57,200.000",63.556,test
524,DEM_00525,18,450.000,"44,000.000",97.778,train


These records have an Annual Value per sq ft far outside the usual range (beyond 3 IQR on the log scale).
Possible causes are entry errors, partly non-residential use, or a floor area that does not match the assessed unit.
They are **kept** in the main analysis, because the data must not be cleaned to make the model look better. A
sensitivity check without them is reported in notebook 03.

## 3. Model A predictors and validation design

In [10]:
sets, screened, screen_table = M.feature_sets(train)
categories = M.condition_categories(lab["Condition"])
pd.DataFrame([{"Algorithm": C.MODEL_NAMES[k], "Model A predictors": ", ".join(sets[("A", k)])} for k in C.ALGORITHMS])

,Algorithm,Model A predictors
0,Multiple Linear Regression,"Age, Condition, Log_Total_Floor_Area, Num_Floors"
1,Random Forest,"Age, Condition, Total_Floor_Area, Num_Floors"
2,XGBoost,"Age, Condition, Total_Floor_Area, Num_Floors"


* **MLR** uses log(floor area). With log(Annual Value) as the target, this is the standard log-log hedonic form
  (Annual Value ~ rate x area), and its coefficient is an elasticity. Condition is dummy-coded with **Fair** as the
  reference grade.
* **RF / XGBoost** use floor area in sq ft (tree models are not affected by monotone transforms). Condition is an
  ordinal grade: Poor < Fair- < Fair < Fair+ < Good.
* Hyperparameters are tuned with `RandomizedSearchCV` (5-fold CV on the training set; RF: 40 candidates, XGBoost:
  60 candidates, scored by RMSE in rupees).

In [11]:
print(f"Training records: {len(train)}, test records: {len(test)}")
pd.concat([train[C.TARGET].describe().rename("train"), test[C.TARGET].describe().rename("test")], axis=1)

Training records: 401, test records: 101


,train,test
count,401.000,101.000
mean,"26,060.948","25,617.347"
std,"29,438.149","24,022.845"
min,"2,640.000","1,152.000"
25%,"9,600.000","8,640.000"
50%,"17,280.000","16,320.000"
75%,"33,600.000","30,720.000"
max,"307,200.000","115,200.000"


## 4. Train, tune and validate Model A (MLR, RF, XGBoost)

In [12]:
outA = R.run_set("A", data, sets, categories, permutation=True)
perfA, tuneA, ratioA = R.summary_tables(outA)

Model A - property only | Multiple Linear Regression  CV RMSE   12,417 CV R2 pooled 0.702 | test RMSE    6,626 R2 0.923 | 8s


Model A - property only | Random Forest               CV RMSE   11,610 CV R2 pooled 0.800 | test RMSE    6,906 R2 0.917 | 18s


Model A - property only | XGBoost                     CV RMSE   11,663 CV R2 pooled 0.792 | test RMSE    6,171 R2 0.933 | 6s


### Cross-validation (training set) and hold-out test performance

In [13]:
cols = ["Algorithm", "CV RMSE", "CV RMSE SD", "CV MAE", "CV R2 pooled", "Train R2", "Test RMSE", "Test MAE", "Test R2"]
perfA[cols]

,Algorithm,CV RMSE,CV RMSE SD,CV MAE,CV R2 pooled,Train R2,Test RMSE,Test MAE,Test R2
0,Multiple Linear Regression,"12,417.444","10,289.739","4,104.119",0.702,0.745,"6,626.339","3,626.702",0.923
1,Random Forest,"11,609.659","6,224.479","3,995.526",0.800,0.907,"6,905.540","3,406.758",0.917
2,XGBoost,"11,663.288","6,722.302","4,042.048",0.792,0.899,"6,171.024","3,143.016",0.933


In [14]:
tuneA

,Model set,Algorithm,Predictors,Predictor list,Tuned hyperparameters
0,Model A - property only,Multiple Linear Regression,4,"Age, Condition, Log_Total_Floor_Area, Num_Floors",none (OLS)
1,Model A - property only,Random Forest,4,"Age, Condition, Total_Floor_Area, Num_Floors","{""max_depth"": 15, ""max_features"": 1.0, ""min_sa..."
2,Model A - property only,XGBoost,4,"Age, Condition, Total_Floor_Area, Num_Floors","{""subsample"": 0.7, ""reg_lambda"": 10, ""reg_alph..."


In [15]:
show(P.actual_vs_predicted(test[C.TARGET], {k: r["pred_test"] for k, r in outA["results"].items()},
                           C.MODEL_SETS["A"]), "A_actual_vs_predicted.png")

## 5. MLR coefficients (statistical interpretation of Model A)

In [16]:
olsA, olsA_table, olsA_stats = M.ols_fit(train, sets[("A", "mlr")], categories)
olsA_table.to_csv(T / "A_mlr_ols.csv", index=False)
pd.DataFrame([olsA_stats]).to_csv(T / "A_mlr_ols_fit.csv", index=False)
display(pd.DataFrame([olsA_stats]))
olsA_table[["Term", "Coefficient (log scale)", "Std. error", "p-value", "Readable effect"]]

,n,R2 (log scale),Adj. R2 (log scale),F p-value,AIC,Condition number
0,401,0.932,0.931,0.000,-49.766,305.304


,Term,Coefficient (log scale),Std. error,p-value,Readable effect
0,const,3.508,0.114,0.000,
1,Age,-0.001,0.001,0.279,-1.25 % per +10 years
2,Log_Total_Floor_Area,0.980,0.019,0.000,+9.79 % per +10 % floor area (elasticity 0.980)
3,Num_Floors,-0.102,0.019,0.000,-9.68 % per extra floor (same total area)
4,Condition_Poor,-0.376,0.118,0.002,-31.32 % vs Fair
5,Condition_Fair-,-0.257,0.061,0.000,-22.70 % vs Fair
6,Condition_Fair+,0.064,0.029,0.025,+6.65 % vs Fair
7,Condition_Good,0.146,0.041,0.000,+15.77 % vs Fair


## 6. Assessment-ratio analysis for Model A (ratio = model estimate / CMC Annual Value)

In [17]:
ratioA[["Algorithm", "Sample", "n", "Median ratio", "COD (%)", "PRD", "PRB", "Within +/-10% (%)", "IAAO check"]]

,Algorithm,Sample,n,Median ratio,COD (%),PRD,PRB,Within +/-10% (%),IAAO check
0,Multiple Linear Regression,Hold-out test,101,1.010,23.127,1.177,-0.139,50.495,level OK; COD > 15; PRD regressive
1,Multiple Linear Regression,Training (out-of-fold CV),401,1.001,14.376,1.052,-0.005,54.863,level OK; uniformity OK; PRD regressive
2,Random Forest,Hold-out test,101,1.001,23.190,1.158,-0.123,63.366,level OK; COD > 15; PRD regressive
3,Random Forest,Training (out-of-fold CV),401,1.003,12.907,1.050,-0.014,59.102,level OK; uniformity OK; PRD regressive
4,XGBoost,Hold-out test,101,1.015,20.279,1.151,-0.122,60.396,level OK; COD > 15; PRD regressive
5,XGBoost,Training (out-of-fold CV),401,0.994,12.943,1.058,-0.020,60.848,level OK; uniformity OK; PRD regressive


IAAO reference ranges (Standard on Ratio Studies, residential): median ratio 0.90-1.10, COD <= 15, PRD 0.98-1.03,
PRB between -0.05 and +0.05. A PRD above 1.03 or a negative PRB means **regressivity**: lower-value properties are
estimated relatively higher than higher-value ones. These measure agreement with the CMC assessments, not
market-value accuracy. All three models are interpreted together with Model B in notebook 03.

In [18]:
assert len(train) + len(test) == len(lab)
assert set(outA["test_ids"]) == set(pd.read_csv(C.SPLIT_FILE).query("Partition == 'test'")["Property_ID"])
print("Model A complete - outputs saved with prefix A_ in outputs/")

Model A complete - outputs saved with prefix A_ in outputs/
